<a href="https://colab.research.google.com/github/Muhammad-Taaha/AI-CHAT-BOT-WITH-VOICE-ASSISTANCE-/blob/main/network_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data collection and info gathering

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import glob

folder_path = "/content/drive/MyDrive/networkproject/archive (1)/"
csv_files = glob.glob(folder_path + "*.csv")

df_list = []
for file in csv_files:
    try:
        df = pd.read_csv(file, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(file, encoding='ISO-8859-1')  # fallback
    df_list.append(df)
    print(f"✅ Loaded: {file} → {df.shape}")

In [ ]:
!head -n 5 "/content/drive/MyDrive/networkproject/archive (1)/UNSW-NB15_1.csv"


In [ ]:
import pandas as pd

features_path = "/content/drive/MyDrive/networkproject/archive (1)/NUSW-NB15_features.csv"
features = pd.read_csv(features_path, encoding='latin1')  # or encoding='ISO-8859-1'
features.head()



In [ ]:
data_path = "/content/drive/MyDrive/networkproject/archive (1)/UNSW-NB15_1.csv"

# Tell pandas there’s no header in the raw file and use the feature names
data = pd.read_csv(data_path, header=None)
data.columns = features['Name']  # use feature column names

# display the full interactive sheet-style table
from google.colab import data_table
data_table.enable_dataframe_formatter()
data.head(50)


In [ ]:
data.columns

In [ ]:
import pandas as pd
import glob

# === 1. Paths ===
folder_path = "/content/drive/MyDrive/networkproject/archive (1)/"
features_path = folder_path + "NUSW-NB15_features.csv"

# === 2. Load column names ===
features = pd.read_csv(features_path, encoding="latin1", na_values=['?', ' ', '', 'NaN', 'nan', 'NULL', 'none'])
column_names = features["Name"].tolist()

# === 3. Get all 4 dataset files ===
csv_files = sorted(glob.glob(folder_path + "UNSW-NB15_[1-4].csv"))

print("Found files:")
for f in csv_files:
    print("📄", f)

# === 4. Read and combine all ===
df_list = []
for f in csv_files:
    print(f"Loading: {f}")
    df = pd.read_csv(f, names=column_names, skiprows=1, encoding="latin1")
    print("✅ Loaded:", f, "→", df.shape)
    df_list.append(df)

# Combine all into one
combined_df = pd.concat(df_list, ignore_index=True)
print("\n✅ Combined dataset shape:", combined_df.shape)

# === 5. Preview ===
combined_df.head()


In [ ]:
combined_df

In [ ]:
df = combined_df

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
df.describe()

In [ ]:
(df.isna().sum() / len(df)) * 100

In [ ]:
df['attack_cat'] = df['attack_cat'].fillna('Normal')

In [ ]:
df['ct_flw_http_mthd'] = df['ct_flw_http_mthd'].fillna(0)

In [ ]:
df['is_ftp_login'] = df['is_ftp_login'].fillna(0)

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
df.dtypes

In [ ]:
df['proto'].nunique()

In [ ]:
df['proto'].unique()

In [ ]:
proto_counts = df['proto'].value_counts()
rare_protocols = proto_counts[proto_counts < 200].index
df['proto'] = df['proto'].replace(rare_protocols, 'other')

In [ ]:
df['proto'].nunique()

In [ ]:
df['proto'].unique()

In [ ]:
df["attack_cat"].value_counts()

In [ ]:
df["service"].nunique()

In [ ]:
df["state"].nunique()

# EDA

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn
import tensorflow
import warnings
warnings.filterwarnings('ignore')
import plotly
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [ ]:
df.shape

In [ ]:
def encode_categorical_columns(df):
    labelencoder = LabelEncoder()
    categorical_cols = ['proto', 'state', 'service', 'ct_ftp_cmd', 'attack_cat']

    for col in categorical_cols:
        if col in df.columns:
            df[col] = df[col].astype(str)  # ensure string type
            df[col] = labelencoder.fit_transform(df[col])

    return df

# Apply
df = encode_categorical_columns(df)


In [ ]:
numeric_df = df.select_dtypes(include=['int64', 'float64'])
corr = numeric_df.corr()
plt.figure(figsize=(10, 12))
sns.heatmap(corr, cmap='coolwarm', annot=False)
plt.title("Correlation Matrix (All Numeric Features)", fontsize=14)
plt.show()

In [ ]:
def top_correlations(df, limit=0.9):
    # Compute correlation matrix
    corr_matrix = df.corr(numeric_only=True)

    # Take absolute values and unstack to get pairs
    corr_pairs = corr_matrix.abs().unstack()

    # Drop self correlations (where feature is correlated with itself)
    corr_pairs = corr_pairs[corr_pairs < 1]

    # Filter based on threshold
    strong_corr = corr_pairs[corr_pairs > limit].sort_values(ascending=False)

    return strong_corr


In [ ]:
strong_corrs = top_correlations(df, limit=0.9)
print(strong_corrs)

In [ ]:
df.dtypes

In [ ]:
# List of features with high correlation (>0.9) from your data
high_corr_features = [
    'Stime', 'Ltime', 'dwin', 'swin', 'dloss', 'Dpkts', 'dbytes',
    'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_srv_src', 'ct_srv_dst',
    'sloss', 'sbytes', 'ct_dst_src_ltm', 'tcprtt',
    'synack', 'ackdat', 'sttl', 'ct_state_ttl', 'Label', 'ct_dst_sport_ltm'
]

# Select only these columns
df_high_corr = df[high_corr_features].sample(n=1000, random_state=42)


In [ ]:
sns.pairplot(df_high_corr, hue='Label', corner=True, plot_kws={'alpha':0.3,'s':20})
plt.show()